<a href="https://colab.research.google.com/github/Janhviiii-19/World-Happiness-Dashboard---GenAI/blob/main/World%20Happiness%20Report%20Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# 1. Read the CSV file into a pandas DataFrame, using the first row as headers
file_path = "/content/drive/MyDrive/Colab Notebooks/2016.csv"
df = pd.read_csv(file_path, header=0)

# 2. Print the first 5 rows to verify correct loading
print(df.head())

       Country          Region  Happiness Rank  Happiness Score  \
0      Denmark  Western Europe               1            7.526   
1  Switzerland  Western Europe               2            7.509   
2      Iceland  Western Europe               3            7.501   
3       Norway  Western Europe               4            7.498   
4      Finland  Western Europe               5            7.413   

   Lower Confidence Interval Upper Confidence Interval  \
0                      7.460                     7.592   
1                      7.428                      7.59   
2                      7.333                     7.669   
3                      7.421                     7.575   
4                      7.351                     7.475   

  Economy (GDP per Capita)   Family Health (Life Expectancy)  Freedom  \
0                  1.44178  1.16374                  0.79504  0.57941   
1                  1.52733  1.14524                  0.86303  0.58557   
2                  1.42666  1

In [2]:
# 1. Check the data types of each column in the DataFrame
print(df.dtypes)

# Optional: get a more detailed summary (dtypes + non-null counts) in one view
print("\nDetailed info:")
df.info()

# Optional: if you want to double check a specific column's actual values
# vs. what pandas inferred, you can inspect a sample like this:
# print(df['column_name'].unique()[:10])

Country                           object
Region                            object
Happiness Rank                     int64
Happiness Score                  float64
Lower Confidence Interval        float64
Upper Confidence Interval         object
Economy (GDP per Capita)          object
Family                           float64
Health (Life Expectancy)          object
Freedom                           object
Trust (Government Corruption)    float64
Generosity                       float64
Dystopia Residual                float64
dtype: object

Detailed info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        157 non-null    object 
 1   Region                         157 non-null    object 
 2   Happiness Rank                 157 non-null    int64  
 3   Happiness Score            

In [3]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. Remove leading/trailing whitespace from all string (object) columns
# ------------------------------------------------------------------
# Select columns that are of 'object' dtype (typically strings)
str_cols = df.select_dtypes(include="object").columns

for col in str_cols:
    df[col] = df[col].str.strip()

# ------------------------------------------------------------------
# 2. Replace empty strings with NaN (pd.NA is the modern, dtype-agnostic
#    "missing value" marker recommended in recent pandas versions)
# ------------------------------------------------------------------
df[str_cols] = df[str_cols].replace("", pd.NA)

# If you only want to do this for ONE specific column, e.g. 'city':
# df['city'] = df['city'].str.strip().replace("", pd.NA)

# ------------------------------------------------------------------
# 3. Convert columns to the most appropriate dtype automatically
# ------------------------------------------------------------------
# convert_dtypes() is the recommended modern approach — it inspects each
# column's actual content and converts to pandas' nullable dtypes
# (Int64, Float64, boolean, string) instead of legacy numpy dtypes,
# which handle missing values (pd.NA) far more gracefully than
# numpy's int64/float64/object.
df = df.convert_dtypes()

# ------------------------------------------------------------------
# Verify the results
# ------------------------------------------------------------------
print(df.dtypes)
print("\n", df.head())

Country                          string[python]
Region                           string[python]
Happiness Rank                            Int64
Happiness Score                         Float64
Lower Confidence Interval               Float64
Upper Confidence Interval        string[python]
Economy (GDP per Capita)         string[python]
Family                                  Float64
Health (Life Expectancy)         string[python]
Freedom                          string[python]
Trust (Government Corruption)           Float64
Generosity                              Float64
Dystopia Residual                       Float64
dtype: object

        Country          Region  Happiness Rank  Happiness Score  \
0      Denmark  Western Europe               1            7.526   
1  Switzerland  Western Europe               2            7.509   
2      Iceland  Western Europe               3            7.501   
3       Norway  Western Europe               4            7.498   
4      Finland  Western E

In [4]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. Identify columns that have missing values
# ------------------------------------------------------------------
missing_summary = df.isna().sum()
cols_with_missing = missing_summary[missing_summary > 0].index.tolist()

print("Columns with missing values and their counts:")
print(missing_summary[missing_summary > 0])
print("\nColumns with missing values:", cols_with_missing)

# ------------------------------------------------------------------
# 2. Replace missing values with the column mean
#    (only applies to numeric columns — mean doesn't make sense
#    for text/categorical columns)
# ------------------------------------------------------------------
numeric_cols_with_missing = df[cols_with_missing].select_dtypes(include=np.number).columns

for col in numeric_cols_with_missing:
    col_mean = df[col].mean()
    df[col] = df[col].fillna(col_mean)

# ------------------------------------------------------------------
# Verify no missing values remain in those numeric columns
# ------------------------------------------------------------------
print("\nMissing values after imputation:")
print(df[numeric_cols_with_missing].isna().sum())

Columns with missing values and their counts:
Lower Confidence Interval    4
Upper Confidence Interval    3
Economy (GDP per Capita)     2
Health (Life Expectancy)     3
Freedom                      1
dtype: int64

Columns with missing values: ['Lower Confidence Interval', 'Upper Confidence Interval', 'Economy (GDP per Capita)', 'Health (Life Expectancy)', 'Freedom']

Missing values after imputation:
Lower Confidence Interval    0
dtype: int64


In [5]:
import pandas as pd
import plotly.graph_objects as go

# ------------------------------------------------------------------
# 1. Identify the top 10 countries (based on Happiness Rank/Score,
#    since this is the 2016 World Happiness Report dataset)
# ------------------------------------------------------------------
# The 2016.csv file typically has columns like:
# 'Country', 'Happiness Rank', 'Happiness Score',
# 'Economy (GDP per Capita)', 'Health (Life Expectancy)', etc.

# Sort by Happiness Rank (ascending, since rank 1 = happiest) and take top 10
top10 = df.sort_values(by="Happiness Rank", ascending=True).head(10)

# Extract only the columns we need
top10_subset = top10[["Country", "Economy (GDP per Capita)", "Health (Life Expectancy)"]]

print("Top 10 countries — GDP per Capita & Healthy Life Expectancy:")
print(top10_subset)

# ------------------------------------------------------------------
# 2. Create a grouped bar chart with Plotly
# ------------------------------------------------------------------
fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x=top10_subset["Country"],
    y=top10_subset["Economy (GDP per Capita)"],
    name="GDP per Capita"
))

fig1.add_trace(go.Bar(
    x=top10_subset["Country"],
    y=top10_subset["Health (Life Expectancy)"],
    name="Healthy Life Expectancy"
))

fig1.update_layout(
    title="GDP per Capita and Healthy Life Expectancy — Top 10 Happiest Countries (2016)",
    xaxis_title="Country",
    yaxis_title="Value",
    barmode="group",
    legend_title="Metric",
    template="plotly_white"
)

fig1.show()

Top 10 countries — GDP per Capita & Healthy Life Expectancy:
       Country Economy (GDP per Capita) Health (Life Expectancy)
0      Denmark                  1.44178                  0.79504
1  Switzerland                  1.52733                  0.86303
2      Iceland                  1.42666                  0.86733
3       Norway                  1.57744                  0.79579
4      Finland                  1.40598                  0.81091
5       Canada                  1.44015                   0.8276
6  Netherlands                  1.46468                  0.81231
7  New Zealand                  1.36066                  0.83096
8    Australia                  1.44443                   0.8512
9       Sweden                  1.45181                  0.83121


In [6]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# ------------------------------------------------------------------
# 1. Create a sub-dataset with the selected attributes
# ------------------------------------------------------------------
columns_of_interest = [
    "Economy (GDP per Capita)",
    "Family",
    "Health (Life Expectancy)",
    "Freedom",
    "Trust (Government Corruption)",
    "Generosity",
    "Happiness Score"
]

sub_df = df[columns_of_interest]

print("Sub-dataset preview:")
print(sub_df.head())

# ------------------------------------------------------------------
# 2. Compute the correlation matrix and plot as a heatmap
# ------------------------------------------------------------------
corr_matrix = sub_df.corr()

print("\nCorrelation matrix:")
print(corr_matrix)

fig2 = px.imshow(
    corr_matrix,
    text_auto=".2f",           # show correlation values on the heatmap
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,           # correlation always ranges from -1 to 1
    title="Correlation Heatmap: Happiness Score and Contributing Factors"
)

fig2.update_layout(
    width=800,
    height=600,
    template="plotly_white"
)

fig2.show()

Sub-dataset preview:
  Economy (GDP per Capita)   Family Health (Life Expectancy)  Freedom  \
0                  1.44178  1.16374                  0.79504  0.57941   
1                  1.52733  1.14524                  0.86303  0.58557   
2                  1.42666  1.18326                  0.86733  0.56624   
3                  1.57744   1.1269                  0.79579  0.59609   
4                  1.40598  1.13464                  0.81091  0.57104   

   Trust (Government Corruption)  Generosity  Happiness Score  
0                        0.44453     0.36171            7.526  
1                        0.41203     0.28083            7.509  
2                        0.14975     0.47678            7.501  
3                        0.35776     0.37895            7.498  
4                        0.41004     0.25492            7.413  

Correlation matrix:
                               Economy (GDP per Capita)    Family  \
Economy (GDP per Capita)                       1.000000  0.669733 

In [9]:
import pandas as pd
import plotly.express as px

# ------------------------------------------------------------------
# Aggregate Happiness Score by Region (using mean, since summing
# scores across countries isn't a meaningful quantity)
# ------------------------------------------------------------------
region_happiness = df.groupby("Region", as_index=False)["Happiness Score"].mean()

# ------------------------------------------------------------------
# Create the pie chart
# ------------------------------------------------------------------
fig4 = px.pie(
    region_happiness,
    names="Region",
    values="Happiness Score",
    title="Average Happiness Score by Region",
    template="plotly_white"
)

fig4.update_traces(
    textposition="inside",
    textinfo="percent+label"
)

fig4.update_layout(
    width=800,
    height=600
)

fig4.show()

In [15]:
import pandas as pd
import numpy as np
import plotly.express as px

# ------------------------------------------------------------------
# Build a plotting-safe copy of df:
# 1. Cast everything to plain object dtype (drops pandas nullable
#    extension types like Int64/Float64/string/boolean)
# 2. Explicitly replace any remaining missing values (pd.NA, NaN, NaT)
#    with Python None, which orjson CAN serialize
# ------------------------------------------------------------------
df_plot = df.astype(object).where(df.notna(), None)

# ------------------------------------------------------------------
# fig3 — Scatter plot: Happiness Score vs. GDP per Capita, by Region
# ------------------------------------------------------------------
fig3 = px.scatter(
    df_plot,
    x="Economy (GDP per Capita)",
    y="Happiness Score",
    color="Region",
    hover_name="Country",
    title="Happiness Score vs. GDP per Capita, by Region",
    labels={
        "Economy (GDP per Capita)": "GDP per Capita",
        "Happiness Score": "Happiness Score"
    },
    template="plotly_white"
)

fig3.update_layout(width=900, height=600, legend_title="Region")
fig3.show()

# ------------------------------------------------------------------
# fig5 — Choropleth map: GDP per Capita by country,
# with Healthy Life Expectancy shown on hover
# ------------------------------------------------------------------
fig5 = px.choropleth(
    df_plot,
    locations="Country",
    locationmode="country names",
    color="Economy (GDP per Capita)",
    hover_name="Country",
    hover_data={
        "Economy (GDP per Capita)": ":.3f",
        "Health (Life Expectancy)": ":.3f",
        "Country": False
    },
    color_continuous_scale="Viridis",
    title="GDP per Capita by Country (Healthy Life Expectancy shown on hover)"
)

fig5.update_layout(
    width=1000,
    height=600,
    template="plotly_white",
    coloraxis_colorbar=dict(title="GDP per Capita")
)
fig5.show()

In [13]:
import plotly.io as pio

# ------------------------------------------------------------------
# Write any four of the figures (fig1, fig2, fig3, fig5 chosen here)
# into a single HTML file, one after another
# ------------------------------------------------------------------
figures_to_include = [fig1, fig2, fig3, fig5]

with open("dashboard.html", "w", encoding="utf-8") as f:
    # Write the first figure with its own copy of the Plotly.js library
    f.write(pio.to_html(figures_to_include[0], include_plotlyjs="cdn", full_html=True))

    # Append the remaining figures as HTML fragments (no need to
    # re-include the Plotly.js library or repeat <html>/<body> tags)
    for fig in figures_to_include[1:]:
        f.write(pio.to_html(fig, include_plotlyjs=False, full_html=False))

print("dashboard.html has been created with 4 figures.")

dashboard.html has been created with 4 figures.


# World Happiness Report Dashboard: A Narrative

## Introduction

What actually makes a country happy? The World Happiness Report has asked this question every year since 2012, and the 2016 edition offers a rich lens into how economic prosperity, health, and regional context intertwine to shape a population's sense of well-being. This dashboard brings together four complementary views of that data — a correlation heatmap, a regional scatter plot, a pie chart, and a world map — to build a layered understanding of what drives happiness across the globe.

## 1. What Actually Correlates with Happiness? (Correlation Heatmap)

We begin with the numbers behind the story. The correlation heatmap lays out the relationships between Happiness Score and six key contributing factors: GDP per Capita, Family, Health (Life Expectancy), Freedom, Trust in Government, and Generosity.

The pattern that emerges is striking: **GDP per Capita, Family, and Health (Life Expectancy) show the strongest positive correlations with Happiness Score**, suggesting that a country's material wealth, social support systems, and public health outcomes move closely together. Freedom also correlates meaningfully with happiness, while Trust (Government Corruption) and Generosity tend to show weaker relationships — a reminder that happiness isn't purchased by wealth alone, but wealth clearly buys the foundation on which other well-being factors are built.

This heatmap sets the stage for the rest of the dashboard: it tells us *which* variables are worth digging into further, and GDP per Capita immediately stands out as a variable worth a closer look.

## 2. Does Money Buy Happiness? It Depends Where You Live (Scatter Plot)

The scatter plot takes the strongest signal from the heatmap — GDP per Capita — and puts it under the microscope, plotting it against Happiness Score for every country, colored by Region.

The overall trend is clearly upward: **as GDP per Capita increases, so does Happiness Score.** But the real story is in the regional clustering. Western Europe and North America sit consistently in the upper-right — high GDP, high happiness. Meanwhile, regions like Sub-Saharan Africa cluster toward the lower-left, with both lower GDP and lower happiness scores. Interestingly, some regions — like Latin America — punch above their economic weight, landing at higher happiness levels than their GDP alone would predict, hinting that cultural and social factors (perhaps the "Family" and "Freedom" variables from the heatmap) are compensating for lower material wealth.

This chart makes clear that **GDP per Capita is a strong predictor of happiness, but not the whole story** — geography and regional context visibly bend the relationship.

## 3. Where Does Happiness Concentrate? (Pie Chart)

Zooming out from individual countries, the pie chart aggregates average Happiness Score by Region, giving a bird's-eye view of how well-being is distributed across the world.

Here, the regional disparities become even more visible in proportional terms. Regions like **Western Europe, North America, and Australia/New Zealand** claim disproportionately large slices relative to their share of countries, while regions such as **Sub-Saharan Africa and Southern Asia** occupy comparatively smaller shares of the overall happiness "pie." This visualization reinforces a central theme of the report: happiness is not evenly distributed geographically, and the gap between the highest- and lowest-scoring regions remains substantial.

## 4. Putting Happiness on the Map (Choropleth Map)

Finally, the choropleth map grounds all of this in geography, shading every country by its GDP per Capita — with Healthy Life Expectancy available on hover for a quick two-metric comparison, country by country.

The map makes the economic geography of happiness immediately visible: a visibly darker (or brighter, depending on the color scale) band of high-GDP countries clusters around North America, Western Europe, and parts of East Asia and Oceania, while lower GDP per capita is concentrated across much of Africa and parts of South Asia. Hovering over any country instantly surfaces its Healthy Life Expectancy alongside its GDP, letting the viewer explore firsthand how closely these two metrics tend to travel together — a country's wealth and its citizens' health outcomes are rarely far apart.

## Bringing It Together

Across all four visuals, one narrative reinforces itself from different angles: **economic prosperity, health, and social support are deeply intertwined with national happiness, but regional and cultural context meaningfully shapes how that relationship plays out.** The heatmap tells us *what* matters statistically; the scatter plot shows *how* that plays out with regional nuance; the pie chart reveals *where* happiness concentrates globally; and the map lets us explore *which specific countries* sit at the extremes. Together, they move beyond a single number — the Happiness Score — into a fuller picture of what well-being actually looks like around the world in 2016.